In [2]:
import torch
import random
import evaluate
import os
from torch.utils.data import DataLoader
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    DataCollatorForTokenClassification, 
    AutoConfig, 
    set_seed
)
from tqdm.auto import tqdm

# Set seed and environment
set_seed(42)
os.environ["HF_TOKEN"] = "your_token_here" # It's better to use an env variable than hardcoding!

# 1. Load the BC5CDR dataset directly from Hugging Face
print("Loading BC5CDR dataset...")
raw_datasets = load_dataset("tner/bc5cdr", trust_remote_code=True)

# Define hyperparameters
learning_rate = 2e-5
num_train_epochs = 3
model_name = "google-bert/bert-base-cased"
batch_size = 8

# 2. Extract label information manually
print("Extracting unique tags from dataset...")

# Collect all unique tag IDs from the training set
unique_tag_ids = set()
for example in raw_datasets["train"]:
    unique_tag_ids.update(example["tags"])

# Convert IDs to a sorted list
# Note: In tner/bc5cdr, these are usually already integers (0, 1, 2, 3, 4)
sorted_tag_ids = sorted(list(unique_tag_ids))

# BC5CDR mapping for this dataset version:
# 0: O, 1: B-Chemical, 2: I-Chemical, 3: B-Disease, 4: I-Disease
label_list = ["O", "B-Chemical", "I-Chemical", "B-Disease", "I-Disease"]

# If the dataset uses more or different tags, you can use this fallback:
if len(sorted_tag_ids) != len(label_list):
    label_list = [str(i) for i in sorted_tag_ids]

tag2id = {tag: i for i, tag in enumerate(label_list)}
id2tag = {i: tag for i, tag in enumerate(label_list)}

text_column_name = "tokens"
label_column_name = "tags"



# Read tokenizer and model configuration
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
config = AutoConfig.from_pretrained(
    model_name, 
    num_labels=len(label_list), 
    id2label=id2tag, 
    label2id=tag2id
)

# 3. Tokenization and Alignment logic (remains mostly the same)
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples[text_column_name],
        max_length=128,             
        padding=False,              
        truncation=True, 
        is_split_into_words=True
    )

    all_labels = []
    for batch_index, labels in enumerate(examples[label_column_name]):
        word_ids = tokenized_inputs.word_ids(batch_index=batch_index)
        label_ids = []
        prev_word_id = None
        
        for word_id in word_ids:
            if word_id is None:
                label_ids.append(-100) 
            elif word_id == prev_word_id:
                label_ids.append(-100)
            else:
                label_ids.append(labels[word_id])
            prev_word_id = word_id
        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

# Process the dataset
processed_raw_datasets = raw_datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Running tokenizer"
)

# Create Dataloaders
train_dataset = processed_raw_datasets["train"]
eval_dataset = processed_raw_datasets["validation"]
test_dataset = processed_raw_datasets["test"]

model = AutoModelForTokenClassification.from_pretrained(model_name, config=config)
data_collator = DataCollatorForTokenClassification(tokenizer)

train_dataloader = DataLoader(train_dataset, shuffle=True, collate_fn=data_collator, batch_size=batch_size)
eval_dataloader = DataLoader(eval_dataset, collate_fn=data_collator, batch_size=batch_size)
test_dataloader = DataLoader(test_dataset, collate_fn=data_collator, batch_size=batch_size)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# --- Training Loop ---
print("Starts training...")
for epoch in range(num_train_epochs):
    model.train()
    pbar = tqdm(train_dataloader, desc=f"Training Epoch {epoch+1}")
    for batch in pbar:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        pbar.set_postfix({"loss": loss.item()})

# --- Evaluation ---
metric = evaluate.load("seqeval")

def get_labels(predictions, references):
    true_predictions = []
    true_labels = []
    for pred_seq, ref_seq in zip(predictions, references):
        current_preds = [id2tag[p.item()] for p, r in zip(pred_seq, ref_seq) if r != -100]
        current_refs = [id2tag[r.item()] for p, r in zip(pred_seq, ref_seq) if r != -100]
        true_predictions.append(current_preds)
        true_labels.append(current_refs)
    return true_predictions, true_labels

print("Evaluating...")
model.eval()
all_predictions = []
all_labels = []

for batch in tqdm(eval_dataloader, desc="Evaluating"):
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)
    
    predictions = outputs.logits.argmax(dim=-1)
    predicted_labels, true_labels = get_labels(predictions, batch["labels"])
    all_predictions.extend(predicted_labels)
    all_labels.extend(true_labels)

results = metric.compute(predictions=all_predictions, references=all_labels)
print(f"Validation Results: {results['overall_f1']}")


/Users/joakimandersen/Downloads/enter/envs/ml/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading BC5CDR dataset...
Extracting unique tags from dataset...


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 6792.37it/s]
BertForTokenClassification LOAD REPORT from: google-bert/bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok 

Starts training...


Training Epoch 1:  11%|█         | 70/654 [03:49<31:53,  3.28s/it, loss=0.2]  


KeyboardInterrupt: 

In [1]:
import json
import torch

def bio_to_char_spans(raw_text, tokens, predicted_label_ids, id2label):
    entities = []
    current_entity = None
    current_char_pos = 0
    
    for i, (token, label_id) in enumerate(zip(tokens, predicted_label_ids)):
        label_str = id2label[label_id]
        clean_token = token.replace("##", "")
        
        start_idx = raw_text.lower().find(clean_token.lower(), current_char_pos)
        if start_idx == -1:
            start_idx = current_char_pos
        end_idx = start_idx + len(clean_token)
        current_char_pos = end_idx

        if label_str == "O":
            if current_entity:
                entities.append(current_entity)
                current_entity = None
            continue

        entity_type = label_str.split("-")[1]

        if current_entity and current_entity["label"] == entity_type:
            gap = raw_text[current_entity["end"]:start_idx]
            if label_str.startswith("I-") or gap.strip() in ["", "-"]:
                current_entity["end"] = end_idx
                current_entity["text"] = raw_text[current_entity["start"]:end_idx]
                continue

        if current_entity:
            entities.append(current_entity)
        
        current_entity = {
            "label": entity_type,
            "text": raw_text[start_idx:end_idx],
            "start": start_idx,
            "end": end_idx
        }
                
    if current_entity:
        entities.append(current_entity)
    return entities

def run_task_b2_inference(input_file, output_file, model, tokenizer, id2label):
    with open(input_file, 'r') as f:
        test_data = json.load(f)
    
    final_results = []
    model.eval()
    
    print(f"Processing {input_file}...")
    
    for entry in test_data:
        doc_id = entry["doc_id"]
        raw_text = entry["text"]
        
        inputs = tokenizer(raw_text, return_tensors="pt", truncation=True, max_length=128)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1).squeeze().tolist()
            
        if isinstance(predictions, int):
            predictions = [predictions]
        
        tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"].squeeze().tolist())
        if isinstance(tokens, str):
            tokens = [tokens]
        
        filtered_tokens = []
        filtered_preds = []
        for tok, pred in zip(tokens, predictions):
            if tok not in [tokenizer.cls_token, tokenizer.sep_token, tokenizer.pad_token]:
                filtered_tokens.append(tok)
                filtered_preds.append(pred)
        
        predicted_entities = bio_to_char_spans(raw_text, filtered_tokens, filtered_preds, id2label)
        
        final_results.append({
            "doc_id": doc_id,
            "predicted_entities": predicted_entities
        })

    with open(output_file, 'w') as f:
        json.dump(final_results, f, indent=2)
    
    print(f"Done! Results saved to {output_file}")

# Run inference on easy and hard splits
run_task_b2_inference("project/test_easy.json", "project/bert_predictions_easy.json", model, tokenizer, id2tag)
run_task_b2_inference("project/test_hard.json", "project/bert_predictions_hard.json", model, tokenizer, id2tag)

NameError: name 'model' is not defined